In [1]:
import sys
print(sys.executable)

C:\Florence's\Work Environtment\Magang - Imigrasi\Facial Scores\Trial_1\.venv\Scripts\python.exe


In [2]:
from keras.utils import to_categorical
from keras_preprocessing.image import load_img
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D
import os
import pandas as pd
import numpy as np

### Data Preparation

In [3]:
TRAIN_DIR = 'images/train'
TEST_DIR = 'images/test'

In [4]:
def createdataframe(dir):
    image_paths = []
    labels = []
    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir, label)):
            image_paths.append(os.path.join(dir, label, imagename))
            labels.append(label)
        print(label, "completed")
    return image_paths, labels

In [5]:
train = pd.DataFrame()
train['image'], train['label'] = createdataframe(TRAIN_DIR)
print(train)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed
                                image     label
0            images/train\angry\0.jpg     angry
1            images/train\angry\1.jpg     angry
2           images/train\angry\10.jpg     angry
3        images/train\angry\10002.jpg     angry
4        images/train\angry\10016.jpg     angry
...                               ...       ...
28816  images/train\surprise\9969.jpg  surprise
28817  images/train\surprise\9985.jpg  surprise
28818  images/train\surprise\9990.jpg  surprise
28819  images/train\surprise\9992.jpg  surprise
28820  images/train\surprise\9996.jpg  surprise

[28821 rows x 2 columns]


In [6]:
test = pd.DataFrame()
test['image'], test['label'] = createdataframe(TEST_DIR)
print(test)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed
                              image     label
0       images/test\angry\10052.jpg     angry
1       images/test\angry\10065.jpg     angry
2       images/test\angry\10079.jpg     angry
3       images/test\angry\10095.jpg     angry
4       images/test\angry\10121.jpg     angry
...                             ...       ...
7061  images/test\surprise\9806.jpg  surprise
7062  images/test\surprise\9830.jpg  surprise
7063  images/test\surprise\9853.jpg  surprise
7064  images/test\surprise\9878.jpg  surprise
7065   images/test\surprise\993.jpg  surprise

[7066 rows x 2 columns]


### Data Preprocessing

In [7]:
from tqdm.notebook import tqdm

In [8]:
def extract_features(images):
    features = []
    for image in tqdm(images):
        img = load_img(image, grayscale=True)
        img = np.array(img)
        features.append(img)
    features = np.array(features)
    features = features.reshape(len(features), 48, 48, 1)
    return features

In [9]:
train_features = extract_features(train['image'])
test_features = extract_features(test['image'])

  0%|          | 0/28821 [00:00<?, ?it/s]

C:\Florence's\Work Environtment\Magang - Imigrasi\Facial Scores\Trial_1\.venv\Lib\site-packages\keras_preprocessing\image\utils.py:107: UserWarning: grayscale is deprecated. Please use color_mode = "grayscale"
  warnings.warn('grayscale is deprecated. Please use '


  0%|          | 0/7066 [00:00<?, ?it/s]

### Normalization

In [12]:
x_train = train_features / 255.0
x_test = test_features / 255.0

### Result

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(train['label'])
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])
y_train = to_categorical(y_train, num_classes=7)
y_test = to_categorical(y_test, num_classes=7)

### Building the Neural Network Model

In [16]:
# sequential model
model = Sequential()

# convolutional layers
model.add(Conv2D(128, kernel_size=(3,3), activation='relu', input_shape=(48,48,1)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

# flattening
model.add(Flatten())

# fully connected layers
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))

# output layer
model.add(Dense(7, activation='softmax'))

# model compilation
model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

### Model Training

In [25]:
from keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath="best_emotion_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [27]:
model.fit(x=x_train, y=y_train, batch_size=128, epochs=100, validation_data=(x_test, y_test), callbacks=[checkpoint])

Epoch 1/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 0s 552ms/step - accuracy: 0.6037 - loss: 1.0487
Epoch 1: val_loss improved from None to 1.05569, saving model to best_emotion_model.keras

Epoch 1: finished saving model to best_emotion_model.keras
226/226 ━━━━━━━━━━━━━━━━━━━━ 133s 589ms/step - accuracy: 0.6037 - loss: 1.0487 - val_accuracy: 0.6059 - val_loss: 1.0557
Epoch 2/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.6075 - loss: 1.0399
Epoch 2: val_loss improved from 1.05569 to 1.04332, saving model to best_emotion_model.keras

Epoch 2: finished saving model to best_emotion_model.keras
226/226 ━━━━━━━━━━━━━━━━━━━━ 117s 515ms/step - accuracy: 0.6075 - loss: 1.0399 - val_accuracy: 0.6112 - val_loss: 1.0433
Epoch 3/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 0s 468ms/step - accuracy: 0.6056 - loss: 1.0402
Epoch 3: val_loss did not improve from 1.04332
226/226 ━━━━━━━━━━━━━━━━━━━━ 113s 499ms/step - accuracy: 0.6056 - loss: 1.0402 - val_accuracy: 0.6077 - val_loss: 1.0462
Epoch 4/100
226/226 

In [19]:
model.save("emotion_model_checkpoint.keras")